# Replication Notebook for Ruble Oil-Exchange Rate Linkage Study

In [ ]:
# 'arch' 라이브러리 설치 (GARCH 모델링을 위해 필요)
!pip install arch

In [ ]:
# ==============================================================================
# Replication code for the oil-exchange rate linkage study
# - 대상: RUB, UAH, KRW
# - 분석: 기초통계, 상관관계, ADF, OLS, Granger, GARCH, Chow
# - 그래프: 환율 추이, 롤링 상관관계, GARCH Alpha/Beta
# ==============================================================================


import os
import warnings

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from arch import arch_model
from scipy.stats import f as f_dist

# ==============================================================================
# 0. 환경 설정
# ==============================================================================

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 1200)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.unicode_minus"] = False

OUTPUT_DIR = "outputs"
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)


In [ ]:
print("🚀 [Step 1] 데이터 다운로드 및 가공 시작")


# ==============================================================================
# 1. 데이터 수집 및 전처리
# ==============================================================================

tickers = {
    "Oil": "BZ=F",       # Brent Crude Oil
    "Gas": "TTF=F",     # Dutch TTF Gas
    "VIX": "^VIX",      # CBOE Volatility Index
    "DXY": "DX-Y.NYB",  # US Dollar Index
    "RUB": "RUB=X",     # RUB per USD
    "UAH": "UAH=X",     # UAH per USD
    "KRW": "KRW=X"      # KRW per USD
}

START_DATE = "2019-01-01"
war_start = "2022-02-24"
END_DATE = "2026-05-01"

raw = yf.download(
    list(tickers.values()),
    start=START_DATE,
    end=END_DATE,
    interval="1wk",
    auto_adjust=True,
    progress=True
)

print(f"📌 Analysis period: {START_DATE} to 2026-04-30")

# yfinance MultiIndex 처리
if isinstance(raw.columns, pd.MultiIndex):
    if "Close" in raw.columns.get_level_values(0):
        df = raw["Close"].copy()
    else:
        df = raw.xs("Close", level=0, axis=1).copy()
else:
    df = raw.copy()

# 컬럼명 변경
df = df.rename(columns={v: k for k, v in tickers.items()})

# 필요한 컬럼만 유지
df = df[list(tickers.keys())]

# 결측치 처리
df = df.ffill().bfill().dropna()

# 로그수익률
df_log = np.log(df / df.shift(1)).dropna()

# 표준화 자료: 회귀, Granger, Chow에 사용
df_scaled = (df_log - df_log.mean()) / df_log.std()

# 전쟁 전후 분리
df_pre = df_scaled[df_scaled.index < war_start]
df_post = df_scaled[df_scaled.index >= war_start]

print(f"✅ 데이터 준비 완료. Pre: {len(df_pre)}, Post: {len(df_post)}")
print(f"✅ 전체 표본 수: {len(df_scaled)}")

In [ ]:
# ==============================================================================
# 2. Table 1: 기초통계량
# ==============================================================================

print("\n📋 [Table 1] 기초통계량")

table1_desc = df_log.describe().T[["mean", "std", "min", "max"]]
table1_desc["Skewness"] = df_log.skew()
table1_desc["Kurtosis"] = df_log.kurtosis()
table1_desc = table1_desc.round(3)

print(table1_desc)

table1_desc.to_csv(
    os.path.join(TABLE_DIR, "Table1_descriptive_statistics.csv"),
    encoding="utf-8-sig"
)

In [ ]:
# ==============================================================================
# 3. Table 2: 상관관계 행렬
# ==============================================================================

print("\n📋 [Table 2] 상관관계 행렬")

table2_corr = df_log.corr().round(3)

print(table2_corr)

table2_corr.to_csv(
    os.path.join(TABLE_DIR, "Table2_correlation_matrix.csv"),
    encoding="utf-8-sig"
)

In [ ]:
# ==============================================================================
# 4. Table 3: ADF 단위근 검정
# ==============================================================================

print("\n📋 [Table 3] ADF 단위근 검정")

adf_results = []

for col in df_log.columns:
    adf_stat, p_val, used_lag, nobs, critical_values, icbest = adfuller(df_log[col])
    adf_results.append({
        "Variable": col,
        "ADF-Stat": adf_stat,
        "p-value": p_val,
        "Result": "Stationary" if p_val < 0.05 else "Non-Stationary"
    })

table3_adf = pd.DataFrame(adf_results).round(3)

print(table3_adf)

table3_adf.to_csv(
    os.path.join(TABLE_DIR, "Table3_adf_unit_root_test.csv"),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ==============================================================================
# Table 3-2: KPSS 단위근 검정
# ==============================================================================

from statsmodels.tsa.stattools import kpss

print("\n📋 [Table 4] KPSS 단위근 검정")

kpss_results = []

for col in df_log.columns:
    try:
        stat, p_val, lags, crit = kpss(df_log[col], regression='c', nlags='auto')
        kpss_results.append({
            "Variable": col,
            "KPSS-Stat": stat,
            "p-value": p_val,
            "Result": "Stationary" if p_val > 0.05 else "Non-Stationary"
        })
    except Exception as e:
        kpss_results.append({
            "Variable": col,
            "KPSS-Stat": np.nan,
            "p-value": np.nan,
            "Result": "Error"
        })

table3_kpss = pd.DataFrame(kpss_results).round(3)

print(table3_kpss)

table3_kpss.to_csv(
    os.path.join(TABLE_DIR, "Table4_kpss_test.csv"),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ==============================================================================
# 5. Table 4: OLS 회귀분석
# ==============================================================================

print("\n📋 [Table 5] OLS 회귀분석 결과")

countries = ["RUB", "UAH", "KRW"]
x_cols = ["Oil", "Gas", "DXY", "VIX"]

ols_results = []

for ctry in countries:
    for period_name, data in [("Pre-War", df_pre), ("Post-War", df_post)]:
        X = sm.add_constant(data[x_cols])
        y = data[ctry]

        model = sm.OLS(y, X).fit()

        ols_results.append({
            "Country": ctry,
            "Period": period_name,
            "Oil_Coef": model.params["Oil"],
            "Sign": "+" if model.params["Oil"] > 0 else "-",
            "Oil_p-value": model.pvalues["Oil"],
            "Adj_R2": model.rsquared_adj
        })

table4_ols = pd.DataFrame(ols_results).round(3)

print(table4_ols)

table4_ols.to_csv(
    os.path.join(TABLE_DIR, "Table5_ols_results.csv"),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
ols_full_results = []

for ctry in countries:
    for period_name, data in [("Pre-War", df_pre), ("Post-War", df_post)]:
        X = sm.add_constant(data[x_cols])
        y = data[ctry]

        model = sm.OLS(y, X).fit()

        row = {
            "Country": ctry,
            "Period": period_name,
            "Adj_R2": model.rsquared_adj,
            "N": int(model.nobs)
        }

        for var in ["const"] + x_cols:
            row[f"{var}_coef"] = model.params[var]
            row[f"{var}_p"] = model.pvalues[var]

        ols_full_results.append(row)

table_ols_full = pd.DataFrame(ols_full_results).round(3)

print("\n[Appendix Table A1] Full OLS Regression Results")
print(table_ols_full)

table_ols_full.to_csv(
    os.path.join(TABLE_DIR, "Appendix_Table_A1_full_ols_results.csv"),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ==============================================================================
# 6. Table 5: Granger 인과성 검정
# ==============================================================================

print("\n📋 [Table 6] Granger 인과성 검정 결과: Oil -> Currency")

granger_results = []

for ctry in countries:
    for period_name, data in [("Pre", df_pre), ("Post", df_post)]:
        try:
            # grangercausalitytests는 [종속변수, 원인변수] 순서
            gc = grangercausalitytests(
                data[[ctry, "Oil"]],
                maxlag=[1],
                verbose=False
            )

            p_val = gc[1][0]["ssr_chi2test"][1]

            granger_results.append({
                "Country": ctry,
                "Period": period_name,
                "p-value": p_val,
                "Result": "Significant" if p_val < 0.05 else "Not significant"
            })

        except Exception as e:
            granger_results.append({
                "Country": ctry,
                "Period": period_name,
                "p-value": np.nan,
                "Result": f"Error: {str(e)}"
            })

table5_granger = pd.DataFrame(granger_results).round(3)

print(table5_granger)

table5_granger.to_csv(
    os.path.join(TABLE_DIR, "Table6_granger_causality.csv"),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ==============================================================================
# 7. Table 7: GARCH(1,1) 변동성 분석
# ==============================================================================

print("\n📋 [Table 7] GARCH(1,1) 분석 결과")

garch_results = []

# GARCH는 일반적으로 수익률을 100배 스케일링
returns = df_log * 100

for ctry in countries:
    for period_name, data in [
        ("Pre", returns[returns.index < war_start]),
        ("Post", returns[returns.index >= war_start])
    ]:
        try:
            model = arch_model(
                data[ctry],
                vol="Garch",
                p=1,
                q=1,
                mean="Constant",
                dist="normal"
            )

            res = model.fit(disp="off")

            garch_results.append({
                "Country": ctry,
                "Period": period_name,
                "Alpha": res.params.get("alpha[1]", np.nan),
                "Beta": res.params.get("beta[1]", np.nan),
                "Omega": res.params.get("omega", np.nan),
                "LogLik": res.loglikelihood
            })

        except Exception as e:
            garch_results.append({
                "Country": ctry,
                "Period": period_name,
                "Alpha": np.nan,
                "Beta": np.nan,
                "Omega": np.nan,
                "LogLik": np.nan
            })

table6_garch = pd.DataFrame(garch_results).round(3)

print(table6_garch)

table6_garch.to_csv(
    os.path.join(TABLE_DIR, "Table7_garch_results.csv"),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ==============================================================================
# 8. Table 8: Chow 구조변화 검정
# ==============================================================================

print("\n📋 [Table 8] Chow 구조변화 검정 결과")


def chow_test(data, y_col, x_cols, break_date):
    """
    Chow test for structural break at break_date.

    H0: No structural break
    H1: Structural break exists
    """
    pre = data[data.index < break_date]
    post = data[data.index >= break_date]

    X_full = sm.add_constant(data[x_cols])
    y_full = data[y_col]
    full_model = sm.OLS(y_full, X_full).fit()
    rss_full = np.sum(full_model.resid ** 2)

    X_pre = sm.add_constant(pre[x_cols])
    y_pre = pre[y_col]
    pre_model = sm.OLS(y_pre, X_pre).fit()
    rss_pre = np.sum(pre_model.resid ** 2)

    X_post = sm.add_constant(post[x_cols])
    y_post = post[y_col]
    post_model = sm.OLS(y_post, X_post).fit()
    rss_post = np.sum(post_model.resid ** 2)

    k = len(x_cols) + 1
    n1 = len(pre)
    n2 = len(post)

    numerator = (rss_full - (rss_pre + rss_post)) / k
    denominator = (rss_pre + rss_post) / (n1 + n2 - 2 * k)

    f_stat = numerator / denominator
    p_value = 1 - f_dist.cdf(f_stat, k, n1 + n2 - 2 * k)

    return f_stat, p_value


chow_results = []

for ctry in countries:
    try:
        f_stat, p_value = chow_test(df_scaled, ctry, x_cols, war_start)

        chow_results.append({
            "Country": ctry,
            "F-stat": f_stat,
            "p-value": p_value,
            "Break": "Yes" if p_value < 0.05 else "No"
        })

    except Exception as e:
        chow_results.append({
            "Country": ctry,
            "F-stat": np.nan,
            "p-value": np.nan,
            "Break": f"Error: {str(e)}"
        })

table7_chow = pd.DataFrame(chow_results).round(4)

print(table7_chow)

table7_chow.to_csv(
    os.path.join(TABLE_DIR, "Table8_chow_test.csv"),
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
# ==============================================================================
# 9. Figure 1: 환율 추이
# ==============================================================================

print("\n🎨 [Figure 1] 환율 추이 생성")

norm_fx = df[countries] / df[countries].iloc[0] * 100

plt.figure(figsize=(10, 5))
plt.plot(norm_fx.index, norm_fx["RUB"], label="RUB")
plt.plot(norm_fx.index, norm_fx["UAH"], label="UAH")
plt.plot(norm_fx.index, norm_fx["KRW"], label="KRW")
plt.axvline(pd.to_datetime(war_start), linestyle="--", color="black", label="War Start")

plt.title("Figure 1. Exchange Rate Trends")
plt.xlabel("Date")
plt.ylabel("Index (2019=100)")
plt.legend()
plt.tight_layout()

fig1_path = os.path.join(FIGURE_DIR, "Figure1_exchange_rate_trends.png")
plt.savefig(fig1_path, dpi=300)
plt.show()

In [ ]:
print("\n🎨 [Figure 2] 유가–환율 롤링 상관관계 생성")

window = 20

roll_corr = pd.DataFrame(index=df_scaled.index)
roll_corr["Oil-RUB"] = df_scaled["Oil"].rolling(window).corr(df_scaled["RUB"])
roll_corr["Oil-UAH"] = df_scaled["Oil"].rolling(window).corr(df_scaled["UAH"])
roll_corr["Oil-KRW"] = df_scaled["Oil"].rolling(window).corr(df_scaled["KRW"])

plt.figure(figsize=(10, 5))
plt.plot(roll_corr.index, roll_corr["Oil-RUB"], label="Oil-RUB")
plt.plot(roll_corr.index, roll_corr["Oil-UAH"], label="Oil-UAH")
plt.plot(roll_corr.index, roll_corr["Oil-KRW"], label="Oil-KRW")

plt.axhline(0, color="black", linewidth=1)
plt.axvline(pd.to_datetime(war_start), linestyle="--", color="black", label="War Start")

plt.title("Figure 2. Rolling Correlations between Oil and Exchange Rates")
plt.xlabel("Date")
plt.ylabel("Rolling Correlation")
plt.legend()
plt.tight_layout()

fig2_path = os.path.join(FIGURE_DIR, "Figure2_rolling_correlation.png")
plt.savefig(fig2_path, dpi=300)
plt.show()

In [ ]:
# ==============================================================================
# 11. Figure 2: GARCH Alpha/Beta 비교
# ==============================================================================

print("\n🎨 [Figure 4] GARCH Alpha/Beta 비교 그래프 생성")

garch_plot = table6_garch.copy()
garch_plot["Label"] = garch_plot["Country"] + "-" + garch_plot["Period"]

x = np.arange(len(garch_plot))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width / 2, garch_plot["Alpha"], width, label="Alpha")
plt.bar(x + width / 2, garch_plot["Beta"], width, label="Beta")

plt.xticks(x, garch_plot["Label"], rotation=45)
plt.title("Figure 4. GARCH(1,1) Alpha and Beta by Country and Period")
plt.ylabel("Coefficient")
plt.legend()
plt.tight_layout()

fig3_path = os.path.join(FIGURE_DIR, "Figure4_garch_alpha_beta.png")
plt.savefig(fig3_path, dpi=300)
plt.show()


In [ ]:

# ==============================================================================
# 12. Figure 1: OLS Oil 계수 비교
# ==============================================================================

print("\n🎨 [Figure 3] OLS Oil 계수 비교 그래프 생성")

ols_plot = table4_ols.copy()
ols_plot["Label"] = ols_plot["Country"] + "-" + ols_plot["Period"]

plt.figure(figsize=(10, 5))
plt.bar(ols_plot["Label"], ols_plot["Oil_Coef"])
plt.axhline(0, color="black", linewidth=1)

plt.title("Figure 3. Oil Coefficients from OLS Regressions")
plt.ylabel("Oil Coefficient")
plt.xticks(rotation=45)
plt.tight_layout()

fig4_path = os.path.join(FIGURE_DIR, "Figure3_ols_oil_coefficients.png")
plt.savefig(fig4_path, dpi=300)
plt.show()


# ==============================================================================
# 13. 전체 결과 저장: Excel 통합 파일
# ==============================================================================

print("\n✅ 분석 완료")

In [ ]:
df_panel = df_log.copy()

df_panel["Post"] = (df_panel.index >= "2022-02-24").astype(int)

df_panel["Russia"] = 1  # RUB만 1, 나머지 0으로 별도 구성 필요

df_panel["Interaction"] = (
    df_panel["Oil"] * df_panel["Post"] * df_panel["Russia"]
)

X = sm.add_constant(df_panel[["Oil", "Post", "Russia", "Interaction", "DXY", "Gas", "VIX"]])
y = df_panel["RUB"]  # 또는 패널 형태로 구성

model = sm.OLS(y, X).fit()

print(model.summary())

### Newey–West(HAC) 표준오차

In [ ]:
import statsmodels.api as sm

# 기존 데이터 그대로 사용
X = sm.add_constant(df_panel[["Oil", "Post", "Russia", "Interaction", "DXY", "Gas", "VIX"]])
y = df_panel["RUB"]

# Newey-West (HAC) 적용
model_hac = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':5})

print(model_hac.summary())

In [ ]:
maxlags=5

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Variable": model_hac.params.index,
    "Coef.": model_hac.params.values,
    "Std.Err.": model_hac.bse.values,
    "t-stat": model_hac.tvalues.values,
    "p-value": model_hac.pvalues.values
})

results = results.round(4)

print(results)

In [ ]:
def star(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""

results["Coef."] = results.apply(
    lambda x: f"{x['Coef.']}{star(x['p-value'])}", axis=1
)

print(results)

In [ ]:
# ===============================
# 설정값
# ===============================
START_DATE = "2019-01-01"
END_DATE = "2026-04-30"

import pandas as pd
WAR_START = pd.Timestamp("2022-02-24")

In [ ]:
# ==============================================================================
# 확장 대조군 분석: 원유수입국 vs 자원수출국 vs 전장국
# ==============================================================================

extra_tickers = {
    # Core
    "RUB": "RUB=X",
    "UAH": "UAH=X",
    "KRW": "KRW=X",

    # Oil-importing open economies
    "JPY": "JPY=X",
    "EURUSD": "EURUSD=X",  # 주의: USD per EUR, 나중에 역수 처리

    # Resource-exporting economies
    "CAD": "CAD=X",
    "NOK": "NOK=X",
    "KZT": "KZT=X",

    # Common factors
    "Oil": "BZ=F",
    "Gas": "TTF=F",
    "VIX": "^VIX",
    "DXY": "DX-Y.NYB"
}

raw2 = yf.download(
    list(extra_tickers.values()),
    start=START_DATE,
    end=END_DATE,
    interval="1wk",
    auto_adjust=True,
    progress=True
)

if isinstance(raw2.columns, pd.MultiIndex):
    df2 = raw2["Close"].copy()
else:
    df2 = raw2.copy()

df2 = df2.rename(columns={v: k for k, v in extra_tickers.items()})
df2 = df2[list(extra_tickers.keys())].ffill().bfill().dropna()

# EURUSD는 USD per EUR이므로, local-currency units per U.S. dollar 기준에 맞추기 위해 EUR per USD로 변환
df2["EUR"] = 1 / df2["EURUSD"]
df2 = df2.drop(columns=["EURUSD"])

df2_log = np.log(df2 / df2.shift(1)).dropna()
df2_scaled = (df2_log - df2_log.mean()) / df2_log.std()

currency_cols = ["RUB", "UAH", "KRW", "JPY", "EUR", "CAD", "NOK", "KZT"]
control_cols = ["Oil", "Gas", "DXY", "VIX"]

# 국가 그룹 정의
country_group = {
    "RUB": "Russia",
    "UAH": "Warzone",
    "KRW": "OilImporter",
    "JPY": "OilImporter",
    "EUR": "OilImporter",
    "CAD": "ResourceExporter",
    "NOK": "ResourceExporter",
    "KZT": "ResourceExporter"
}

# ==============================================================================
# 국가별 OLS: Oil 계수 비교
# ==============================================================================

extended_ols = []

for ctry in currency_cols:
    for period_name, data in [
        ("Pre-War", df2_scaled[df2_scaled.index < WAR_START]),
        ("Post-War", df2_scaled[df2_scaled.index >= WAR_START])
    ]:
        try:
            X = sm.add_constant(data[control_cols])
            y = data[ctry]
            model = sm.OLS(y, X).fit()

            extended_ols.append({
                "Country": ctry,
                "Group": country_group[ctry],
                "Period": period_name,
                "Oil_Coef": model.params["Oil"],
                "Oil_p": model.pvalues["Oil"],
                "DXY_Coef": model.params["DXY"],
                "DXY_p": model.pvalues["DXY"],
                "Adj_R2": model.rsquared_adj,
                "N": int(model.nobs)
            })
        except Exception as e:
            extended_ols.append({
                "Country": ctry,
                "Group": country_group[ctry],
                "Period": period_name,
                "Oil_Coef": np.nan,
                "Oil_p": np.nan,
                "DXY_Coef": np.nan,
                "DXY_p": np.nan,
                "Adj_R2": np.nan,
                "N": np.nan
            })

table_extended_ols = pd.DataFrame(extended_ols).round(4)
print(table_extended_ols)

table_extended_ols.to_csv(
    os.path.join(TABLE_DIR, "Robustness_extended_country_ols.csv"),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ==============================================================================
# 확장 패널 상호작용 회귀분석
# Russia vs 전체 대조군
# ==============================================================================

panel_list = []

for ctry in currency_cols:
    tmp = df2_scaled[[ctry, "Oil", "Gas", "DXY", "VIX"]].copy()
    tmp = tmp.rename(columns={ctry: "FX"})
    tmp["Country"] = ctry
    tmp["Group"] = country_group[ctry]
    tmp["Post"] = (tmp.index >= WAR_START).astype(int)
    tmp["Russia"] = 1 if ctry == "RUB" else 0
    tmp["Warzone"] = 1 if ctry == "UAH" else 0
    tmp["ResourceExporter"] = 1 if country_group[ctry] == "ResourceExporter" else 0
    tmp["OilImporter"] = 1 if country_group[ctry] == "OilImporter" else 0
    panel_list.append(tmp)

df_panel_ext = pd.concat(panel_list).dropna()

df_panel_ext["Oil_Post_Russia"] = (
    df_panel_ext["Oil"] * df_panel_ext["Post"] * df_panel_ext["Russia"]
)

X_ext = sm.add_constant(
    df_panel_ext[
        [
            "Oil",
            "Post",
            "Russia",
            "Warzone",
            "ResourceExporter",
            "Oil_Post_Russia",
            "Gas",
            "DXY",
            "VIX"
        ]
    ]
)

y_ext = df_panel_ext["FX"]

# Newey-West / HAC 표준오차
model_ext_hac = sm.OLS(y_ext, X_ext).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 5}
)

print(model_ext_hac.summary())

table_ext_interaction = pd.DataFrame({
    "Variable": model_ext_hac.params.index,
    "Coef.": model_ext_hac.params.values,
    "Std.Err.": model_ext_hac.bse.values,
    "t-stat": model_ext_hac.tvalues.values,
    "p-value": model_ext_hac.pvalues.values
}).round(4)

print(table_ext_interaction)

table_ext_interaction.to_csv(
    os.path.join(TABLE_DIR, "Robustness_extended_interaction_HAC.csv"),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# table_extended_ols 사용
plot_df = table_extended_ols.copy()

# 국가 순서
country_order = ["RUB", "UAH", "KRW", "JPY", "EUR", "CAD", "NOK", "KZT"]

pre = plot_df[plot_df["Period"] == "Pre-War"].set_index("Country").loc[country_order]
post = plot_df[plot_df["Period"] == "Post-War"].set_index("Country").loc[country_order]

x = np.arange(len(country_order))
width = 0.35

plt.figure(figsize=(11, 5))
plt.bar(x - width/2, pre["Oil_Coef"], width, label="Pre-War")
plt.bar(x + width/2, post["Oil_Coef"], width, label="Post-War")

plt.axhline(0, color="black", linewidth=1)
plt.xticks(x, country_order)
plt.ylabel("Oil Coefficient")
plt.title("Figure 5. Oil Coefficients Before and After the War by Country")
plt.legend()
plt.tight_layout()

plt.savefig("Figure5_oil_coefficients_extended_countries.png", dpi=300)
plt.show()

# Revision Robustness Checks for Reviewer Response

The following two cells are **self-contained** and can be run independently in Colab without executing the earlier notebook cells.

- **Appendix Table A3**: OLS robustness check excluding the March–May 2020 COVID-19 oil-price collapse period.
- **Appendix Table A4**: GARCH(1,1) sensitivity check using Student-t errors.

Both cells download the required data directly from Yahoo Finance through `yfinance`, apply the same weekly-frequency and transformation logic, and display the resulting appendix tables as **HTML tables with explicit column headers**. This prevents the column names from being concatenated when the output is copied into Word or Markdown.

No CSV output is required. If a file export is later needed, the optional `to_csv()` lines at the end of each cell can be uncommented.


In [ ]:
# ============================================================
# Appendix Table A3
# Robustness Check Excluding the COVID-19 Oil-Price Collapse Period
# ============================================================
# This cell is self-contained. It can be run independently in Colab.

!pip -q install yfinance pandas numpy statsmodels scipy tabulate

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
from IPython.display import display, Markdown

# -----------------------------
# 1. Configuration
# -----------------------------
START = "2019-01-01"
END = "2026-05-01"          # raw-data download endpoint; covers data through April 2026
WAR_DATE = "2022-02-24"
COVID_START = "2020-03-01"
COVID_END = "2020-05-31"

TICKERS = {
    "Oil": "BZ=F",          # Brent crude oil futures
    "Gas": "TTF=F",        # Dutch TTF natural gas futures
    "VIX": "^VIX",         # CBOE Volatility Index
    "DXY": "DX-Y.NYB",     # U.S. Dollar Index
    "RUB": "RUB=X",        # RUB per USD
    "UAH": "UAH=X",        # UAH per USD
    "KRW": "KRW=X",        # KRW per USD
    "JPY": "JPY=X",        # JPY per USD
    "EURUSD": "EURUSD=X",  # Yahoo quote; converted to EUR per USD direction
    "CAD": "CAD=X",        # CAD per USD
    "NOK": "NOK=X",        # NOK per USD
    "KZT": "KZT=X",        # KZT per USD
}

MAIN_CURRENCIES = ["RUB", "UAH", "KRW"]
SUPPLEMENTARY_CURRENCIES = ["JPY", "EUR", "CAD", "NOK", "KZT"]
REGRESSORS = ["Oil", "Gas", "DXY", "VIX"]

# -----------------------------
# 2. Download and transformation
# -----------------------------
raw = yf.download(
    tickers=list(TICKERS.values()),
    start=START,
    end=END,
    interval="1wk",
    auto_adjust=False,
    progress=False,
    group_by="column",
    threads=True,
)

if raw.empty:
    raise RuntimeError("No data were downloaded. Check internet access and ticker availability.")

if isinstance(raw.columns, pd.MultiIndex):
    field = "Adj Close" if "Adj Close" in raw.columns.get_level_values(0) else "Close"
    prices = raw[field].copy()
else:
    field = "Adj Close" if "Adj Close" in raw.columns else "Close"
    prices = raw[[field]].copy()
    prices.columns = list(TICKERS.values())[:1]

reverse = {v: k for k, v in TICKERS.items()}
prices = prices.rename(columns=reverse)
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices = prices.sort_index().ffill().bfill()

# Convert EUR/USD into USD/EUR direction before log differencing,
# consistent with the "local-currency units per U.S. dollar" convention.
if "EURUSD" in prices.columns:
    prices["EUR"] = 1.0 / prices["EURUSD"]
    prices = prices.drop(columns=["EURUSD"])

prices = prices.where(prices > 0)
data = np.log(prices).diff().dropna(how="any")
ordered_cols = [c for c in ["Oil", "Gas", "VIX", "DXY", "RUB", "UAH", "KRW", "JPY", "EUR", "CAD", "NOK", "KZT"] if c in data.columns]
data = data[ordered_cols]

# -----------------------------
# 3. OLS function
# -----------------------------
def fit_ols(df, currency):
    d = df[[currency] + REGRESSORS].dropna()
    if len(d) < len(REGRESSORS) + 10:
        return {
            "Currency": currency,
            "Oil coefficient": np.nan,
            "Oil p-value": np.nan,
            "Adj. R²": np.nan,
            "N": len(d),
            "Status": "Too few observations",
        }

    y = d[currency]
    X = sm.add_constant(d[REGRESSORS], has_constant="add")
    res = sm.OLS(y, X).fit()

    return {
        "Currency": currency,
        "Oil coefficient": res.params.get("Oil", np.nan),
        "Oil p-value": res.pvalues.get("Oil", np.nan),
        "Adj. R²": res.rsquared_adj,
        "N": int(res.nobs),
        "Status": "OK",
    }

def split_periods(df, exclude_covid=False):
    war = pd.Timestamp(WAR_DATE)
    pre = df.loc[df.index < war].copy()
    post = df.loc[df.index >= war].copy()

    if exclude_covid:
        c0 = pd.Timestamp(COVID_START)
        c1 = pd.Timestamp(COVID_END)
        pre = pre.loc[~((pre.index >= c0) & (pre.index <= c1))].copy()

    return [("Prewar", pre), ("Postwar", post)]

# -----------------------------
# 4. Run baseline and COVID-excluded OLS
# -----------------------------
currencies = [c for c in MAIN_CURRENCIES + SUPPLEMENTARY_CURRENCIES if c in data.columns]

rows = []
for spec_name, exclude_covid in [("Baseline", False), ("COVID-excluded", True)]:
    for period, subset in split_periods(data, exclude_covid=exclude_covid):
        for cur in currencies:
            row = fit_ols(subset, cur)
            row["Specification"] = spec_name
            row["Period"] = period
            row["Excluded window"] = f"{COVID_START} to {COVID_END}" if exclude_covid else ""
            rows.append(row)

full_results = pd.DataFrame(rows)
full_results = full_results[
    ["Specification", "Currency", "Period", "Oil coefficient", "Oil p-value", "Adj. R²", "N", "Excluded window", "Status"]
]

appendix_a3 = full_results[full_results["Specification"] == "COVID-excluded"].copy()
appendix_a3 = appendix_a3.drop(columns=["Specification", "Excluded window", "Status"])

# -----------------------------
# 5. Display table output directly in notebook
# -----------------------------
from IPython.display import display, Markdown, HTML

a3_table = appendix_a3.copy().reset_index(drop=True)

# Manuscript-ready numeric formatting
a3_display = a3_table.copy()
a3_display["Oil coefficient"] = a3_display["Oil coefficient"].map(lambda x: f"{x:.3f}")
a3_display["Oil p-value"] = a3_display["Oil p-value"].map(lambda x: "<0.001" if x < 0.001 else f"{x:.3f}")
a3_display["Adj. R²"] = a3_display["Adj. R²"].map(lambda x: f"{x:.3f}")
a3_display["N"] = a3_display["N"].astype(int).astype(str)

display(Markdown("### Appendix Table A3. Robustness Check Excluding the COVID-19 Oil-Price Collapse Period"))

html_a3 = a3_display.to_html(index=False, escape=False, border=0)
html_a3 = html_a3.replace(
    '<table border="0" class="dataframe">',
    '<table style="border-collapse:collapse; font-size:13px; width:auto;" border="1">'
)
html_a3 = html_a3.replace(
    '<th>',
    '<th style="border:1px solid #999; padding:4px 8px; text-align:center; background-color:#f2f2f2;">'
)
html_a3 = html_a3.replace(
    '<td>',
    '<td style="border:1px solid #999; padding:4px 8px; text-align:center;">'
)
display(HTML(html_a3))

display(Markdown(
    "**Note.** The March–May 2020 COVID-19 oil-price collapse period is excluded from the prewar sample. "
    "The same OLS specification as the main analysis is used. All variables are weekly log returns or rates of change."
))

# Plain-text Markdown version for copying into a manuscript or response letter
print("\nMarkdown table for copy/paste:\n")
print(a3_display.to_markdown(index=False))

# Optional export, if needed later:
# full_results.to_csv("ols_baseline_vs_covid_excluded.csv", index=False, encoding="utf-8-sig")
# appendix_a3.to_csv("appendix_table_a3_covid_excluded_ols.csv", index=False, encoding="utf-8-sig")


In [ ]:
# ============================================================
# Appendix Table A4
# GARCH(1,1) Robustness Check with Student-t Errors
# ============================================================
# This cell is self-contained. It can be run independently in Colab.

!pip -q install yfinance pandas numpy arch tabulate

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model
from IPython.display import display, Markdown

# -----------------------------
# 1. Configuration
# -----------------------------
START = "2019-01-01"
END = "2026-05-01"
WAR_DATE = "2022-02-24"

TICKERS = {
    "RUB": "RUB=X",    # RUB per USD
    "UAH": "UAH=X",    # UAH per USD
    "KRW": "KRW=X",    # KRW per USD
}

CURRENCIES = ["RUB", "UAH", "KRW"]
RETURN_SCALE = 100.0    # Scaling improves numerical stability in GARCH estimation.

# -----------------------------
# 2. Download and transformation
# -----------------------------
raw = yf.download(
    tickers=list(TICKERS.values()),
    start=START,
    end=END,
    interval="1wk",
    auto_adjust=False,
    progress=False,
    group_by="column",
    threads=True,
)

if raw.empty:
    raise RuntimeError("No data were downloaded. Check internet access and ticker availability.")

if isinstance(raw.columns, pd.MultiIndex):
    field = "Adj Close" if "Adj Close" in raw.columns.get_level_values(0) else "Close"
    prices = raw[field].copy()
else:
    field = "Adj Close" if "Adj Close" in raw.columns else "Close"
    prices = raw[[field]].copy()
    prices.columns = list(TICKERS.values())[:1]

reverse = {v: k for k, v in TICKERS.items()}
prices = prices.rename(columns=reverse)
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices = prices.sort_index().ffill().bfill()

prices = prices.where(prices > 0)
data = np.log(prices).diff().dropna(how="any")
data = data[[c for c in CURRENCIES if c in data.columns]]

# -----------------------------
# 3. GARCH function
# -----------------------------
def fit_garch_student_t(series, scale=RETURN_SCALE):
    y = series.dropna() * scale

    if len(y) < 50:
        return {
            "omega": np.nan,
            "alpha": np.nan,
            "beta": np.nan,
            "nu": np.nan,
            "LogLik": np.nan,
            "N": len(y),
            "Converged": False,
            "Boundary flag": "Too few observations",
            "Status": "Too few observations",
        }

    try:
        model = arch_model(
            y,
            mean="Constant",
            vol="GARCH",
            p=1,
            o=0,
            q=1,
            dist="t",
            rescale=False,
        )
        res = model.fit(disp="off", show_warning=False, options={"maxiter": 2000})
        params = res.params

        omega = float(params.get("omega", np.nan))
        alpha = float(params.get("alpha[1]", np.nan))
        beta = float(params.get("beta[1]", np.nan))
        nu = float(params.get("nu", np.nan)) if "nu" in params.index else np.nan

        convergence_flag = getattr(res, "convergence_flag", None)
        converged = True if convergence_flag is None else (convergence_flag == 0)

        flags = []
        if np.isfinite(alpha) and alpha <= 1e-4:
            flags.append("alpha near 0")
        if np.isfinite(beta) and beta >= 0.98:
            flags.append("beta near 1")
        if np.isfinite(alpha) and np.isfinite(beta) and alpha + beta >= 0.999:
            flags.append("alpha+beta near/nonstationary")
        if not converged:
            flags.append(f"convergence_flag={convergence_flag}")

        return {
            "omega": omega,
            "alpha": alpha,
            "beta": beta,
            "nu": nu,
            "LogLik": float(res.loglikelihood),
            "N": int(res.nobs),
            "Converged": bool(converged),
            "Boundary flag": "; ".join(flags),
            "Status": "OK",
        }

    except Exception as exc:
        return {
            "omega": np.nan,
            "alpha": np.nan,
            "beta": np.nan,
            "nu": np.nan,
            "LogLik": np.nan,
            "N": len(y),
            "Converged": False,
            "Boundary flag": "Estimation failed",
            "Status": f"Failed: {type(exc).__name__}: {exc}",
        }

def split_periods(df):
    war = pd.Timestamp(WAR_DATE)
    return [
        ("Prewar", df.loc[df.index < war].copy()),
        ("Postwar", df.loc[df.index >= war].copy()),
    ]

# -----------------------------
# 4. Run Student-t GARCH sensitivity check
# -----------------------------
rows = []
for period, subset in split_periods(data):
    for cur in CURRENCIES:
        if cur not in subset.columns:
            continue
        row = fit_garch_student_t(subset[cur])
        row["Currency"] = cur
        row["Period"] = period
        row["Distribution"] = "Student-t"
        row["Scale"] = RETURN_SCALE
        rows.append(row)

appendix_a4 = pd.DataFrame(rows)
appendix_a4 = appendix_a4[
    ["Currency", "Period", "Distribution", "omega", "alpha", "beta", "nu", "LogLik", "N", "Converged", "Boundary flag", "Status"]
]

# -----------------------------
# 5. Display table output directly in notebook
# -----------------------------
from IPython.display import display, Markdown, HTML

a4_table = appendix_a4.copy().reset_index(drop=True)

# Manuscript-ready numeric formatting
a4_display = a4_table.copy()
for col in ["omega", "alpha", "beta", "nu", "LogLik"]:
    a4_display[col] = a4_display[col].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
a4_display["N"] = a4_display["N"].astype(int).astype(str)
a4_display["Converged"] = a4_display["Converged"].astype(str)

display(Markdown("### Appendix Table A4. GARCH Robustness Check with Student-t Errors"))

html_a4 = a4_display.to_html(index=False, escape=False, border=0)
html_a4 = html_a4.replace(
    '<table border="0" class="dataframe">',
    '<table style="border-collapse:collapse; font-size:13px; width:auto;" border="1">'
)
html_a4 = html_a4.replace(
    '<th>',
    '<th style="border:1px solid #999; padding:4px 8px; text-align:center; background-color:#f2f2f2;">'
)
html_a4 = html_a4.replace(
    '<td>',
    '<td style="border:1px solid #999; padding:4px 8px; text-align:center;">'
)
display(HTML(html_a4))

display(Markdown(
    "**Note.** GARCH(1,1) models are estimated using Student-t errors. "
    "Returns are multiplied by 100 before estimation for numerical stability. "
    "Boundary flags indicate estimates that require cautious interpretation."
))

# Plain-text Markdown version for copying into a manuscript or response letter
print("\nMarkdown table for copy/paste:\n")
print(a4_display.to_markdown(index=False))

# Optional export, if needed later:
# appendix_a4.to_csv("appendix_table_a4_student_t_garch.csv", index=False, encoding="utf-8-sig")
